

## Background
This is my favorite project I learnt from Coursera, it shows me the way how I collect real time data of temperature, humidity of minute window.
Smart Building Solutions, Inc. specializes in optimizing HVAC (heating, ventilation, and air conditioning) systems to enhance comfort and energy efficiency in commercial buildings. By monitoring temperature and humidity levels in real-time across various rooms, the company aims to ensure optimal indoor conditions and preemptively address potential HVAC issues.

With a continuous influx of sensor data, Smart Building Solutions needs to process and analyze this data in real-time to maintain the quality of the indoor environment.

## Data set description
The simulated data set comprises:

`room_id`: Unique identifier for each room (e.g., R001, R002).

`temperature`: Current temperature reading from the sensor (in °C).

`humidity`: Current humidity level reading from the sensor (in %).

`timestamp`: Time when the reading was recorded (automatically generated by Spark).
The data is generated at a rate of 5 rows per second, simulating multiple rooms with various environmental conditions.


## Challenges
Monitoring indoor environmental conditions poses several challenges:

**High data velocity**: Continuous data from multiple sensors can overwhelm traditional systems.

**Need for immediate alerts**: Delays in identifying critical conditions can lead to discomfort or system inefficiencies.

**Need for data aggregation and analysis**: Efficiently aggregating and analyzing real-time data for proactive maintenance and optimization is essential.

## Apache Spark with structured streaming
To address these challenges, Apache Spark is employed for its powerful distributed computing capabilities, enabling real-time data processing and analytics.


In [1]:
!pip install pyspark==3.1.2 -q
!pip install findspark -q

In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# FindSpark simplifies the process of using Apache Spark with Python

import findspark
findspark.init()

#import functions/Classes for sparkml

from pyspark.ml.clustering import KMeans


from pyspark.sql import SparkSession


### Set up the Spark session:


In [3]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Smart Building HVAC Monitoring") \
    .getOrCreate()


25/12/18 16:17:27 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Simulate sensor data:

Use Spark’s rate source to generate continuous readings from multiple rooms.


In [11]:
from pyspark.sql.functions import expr, rand,when

# Simulate sensor data with room IDs and readings
sensor_data = spark.readStream.format("rate").option("rowsPerSecond", 5).load() \
    .withColumn("room_id", expr("CAST(value % 10 AS STRING)")) \
    .withColumn("temperature", when(expr("value % 10 == 0"), 15)  # Set temperature to 15 for one specific record
                .otherwise(20 + rand() * 25)) \
    .withColumn("humidity", expr("40 + rand() * 30"))

### Create a temporary SQL view:

Create temporary SQL view to perform SQL queries on the streaming data.


In [12]:
# Create a temporary SQL view for the sensor data
sensor_data.createOrReplaceTempView("sensor_table")


### Define SQL queries for aggregation and analysis:

* **Critical temperature query**: Detect rooms with critical temperature levels
* **Average readings query**: Calculate average readings over a 1-minute window
* **Attention needed query**: Identify rooms that need immediate attention based on humidity levels


In [13]:
# SQL Query to detect rooms with critical temperatures
critical_temperature_query = """
    SELECT 
        room_id, 
        temperature, 
        humidity, 
        timestamp 
    FROM sensor_table 
    WHERE temperature < 18 OR temperature > 60
"""

# SQL Query to calculate average readings over a 1-minute window
average_readings_query = """
    SELECT 
        room_id, 
        AVG(temperature) AS avg_temperature, 
        AVG(humidity) AS avg_humidity, 
        window.start AS window_start 
    FROM sensor_table
    GROUP BY room_id, window(timestamp, '1 minute')
"""

# SQL Query to find rooms that need immediate attention based on humidity
attention_needed_query = """
    SELECT 
        room_id, 
        COUNT(*) AS critical_readings 
    FROM sensor_table 
    WHERE humidity < 45 OR humidity > 75
    GROUP BY room_id
"""


### Execute the SQL queries:

Execute each SQL query to create streaming DataFrames.


In [14]:
# Execute the critical temperature query
critical_temperatures_stream = spark.sql(critical_temperature_query)


# Execute the average readings query
average_readings_stream = spark.sql(average_readings_query)

# Execute the attention needed query
attention_needed_stream = spark.sql(attention_needed_query)






### Output the results to the console:

Display the results from each query in real-time.


In [15]:
# Output the results to the console for all queries
critical_query = critical_temperatures_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .queryName("Critical Temperatures") \
    .start()

average_query = average_readings_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Average Readings") \
    .start()

attention_query = attention_needed_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Attention Needed") \
    .start()



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----------+-----------------+--------------------+
|room_id|temperature|         humidity|           timestamp|
+-------+-----------+-----------------+--------------------+
|      0|       15.0|63.56153712966483|2025-12-18 16:35:...|
|      0|       15.0|53.54064997006709|2025-12-18 16:35:...|
|      0|       15.0|67.56599767125918|2025-12-18 16:35:...|
+-------+-----------+-----------------+--------------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---------------+------------+------------+
|room_id|avg_temperature|avg_humidity|window_start|
+-------+---------------+------------+------------+
+-------+---------------+------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|48.781985158526666|2025-12-18 16:35:...|
|      0|       15.0| 44.98894033690954|2025-12-18 16:35:...|
|      0|       15.0|  63.9180775161647|2025-12-18 16:35:...|
|      0|       15.0|41.708077557981476|2025-12-18 16:35:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
+-------+-----------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 51.44831349192326|2025-12-18 16:35:...|
|      0|       15.0|44.633511172362695|2025-12-18 16:35:...|
|      0|       15.0| 52.43981636298012|2025-12-18 16:35:...|
|      0|       15.0| 64.09287154557295|2025-12-18 16:36:...|
|      0|       15.0|40.202630840892745|2025-12-18 16:36:...|
|      0|       15.0| 52.98625917135167|2025-12-18 16:35:...|
|      0|       15.0|47.152585086052824|2025-12-18 16:35:...|
|      0|       15.0| 60.78917000940376|2025-12-18 16:36:...|
|      0|       15.0|50.757235218742245|2025-12-18 16:36:...|
|      0|       15.0| 51.37956085984582|2025-12-18 16:36:...|
|      0|       15.0| 42.70341415105523|2025-12-18 16:35:...|
|      0|       15.0| 42.1477496136

...

+-------+------------------+------------------+-------------------+
|room_id|   avg_temperature|      avg_humidity|       window_start|
+-------+------------------+------------------+-------------------+
|      4| 29.88273171017573|51.992787310436675|2025-12-18 16:39:00|
|      1|  32.4613604317811|  55.7469336910981|2025-12-18 16:38:00|
|      1|31.115445456328775|55.288204472659736|2025-12-18 16:37:00|
|      6|35.806961455405684| 51.07310065670765|2025-12-18 16:39:00|
|      4| 33.04076095580852|  56.0983964983261|2025-12-18 16:36:00|
|      8| 32.52679111118005| 54.05917216929594|2025-12-18 16:35:00|
|      9| 32.02578110530474| 54.23622359098051|2025-12-18 16:37:00|
|      8| 32.53952840110149| 53.24095024969301|2025-12-18 16:38:00|
|      3| 32.32753516023977|54.223831714695656|2025-12-18 16:38:00|
|      8| 33.52277174749643| 54.69797596286135|2025-12-18 16:36:00|
|      5| 32.39189357531329| 52.58421243098538|2025-12-18 16:36:00|
|      7|25.596517503434267| 53.25451749146802|2

-------------------------------------------
Batch: 16
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|41.562800806830964|2025-12-18 16:39:...|
|      0|       15.0| 60.75581542022488|2025-12-18 16:39:...|
|      0|       15.0| 68.59650011608018|2025-12-18 16:39:...|
|      0|       15.0| 51.90772382570931|2025-12-18 16:39:...|
|      0|       15.0| 51.73285026567018|2025-12-18 16:39:...|
|      0|       15.0| 57.25628501919703|2025-12-18 16:39:...|
|      0|       15.0|  48.2370301840726|2025-12-18 16:39:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 7
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
|      7|               15|
|      3| 

-------------------------------------------
Batch: 17
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 51.65749477132214|2025-12-18 16:39:...|
|      0|       15.0| 65.33821873255904|2025-12-18 16:39:...|
|      0|       15.0|61.144982534687244|2025-12-18 16:39:...|
|      0|       15.0| 64.64610289527357|2025-12-18 16:39:...|
|      0|       15.0|  59.0385707353656|2025-12-18 16:39:...|
|      0|       15.0| 57.54047696734618|2025-12-18 16:39:...|
|      0|       15.0| 61.42329419337707|2025-12-18 16:39:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 8
-------------------------------------------
+-------+------------------+------------------+-------------------+
|room_id|   avg_temperature|      avg_humidity|       

-------------------------------------------
Batch: 18
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 66.67939200914685|2025-12-18 16:39:...|
|      0|       15.0|60.059761422359244|2025-12-18 16:39:...|
|      0|       15.0| 66.40546356918064|2025-12-18 16:39:...|
|      0|       15.0| 47.90496527910214|2025-12-18 16:39:...|
|      0|       15.0| 50.43048230440091|2025-12-18 16:39:...|
|      0|       15.0| 46.42951482928373|2025-12-18 16:40:...|
|      0|       15.0|52.625776611497955|2025-12-18 16:39:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 8
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
|      7|               17|
|      3| 

[Stage 55:(146 + 8) / 200][Stage 56:>   (0 + 0) / 8][Stage 57:>   (0 + 0) / 8]8]

### Keep the streams running:

Ensure that the streaming queries continue to run to process incoming data.


In [ ]:
# Keep the streams running

print("********Critical Temperature Values*******")
critical_query.awaitTermination()

print("********Average Readings Values********")
average_query.awaitTermination()
print("********Attention Needed Values********")
attention_query.awaitTermination()


### Conclusion
In this lab, you explored the use of Apache Spark in smart building monitoring, particularly focusing on HVAC (heating, ventilation, and air conditioning) systems. You now understand the Spark's distributed architecture. You also understand how to simulate real-time sensor data for temperature and humidity, execute SQL queries to identify critical environmental conditions, and output aggregated results for immediate insights.


## Author(s)

Lakshmi Holla

## Other Contributors
Malika Singla
